In [4]:
import pandas as pd

df = pd.read_csv('tagasiside_log.csv')
print(f'Kokku päringuid: {len(df)}')
df[['Aeg', 'Kasutaja päring', 'Hinnang', 'Veatüüp']]

Kokku päringuid: 8


,Aeg,Kasutaja päring,Hinnang,Veatüüp
0,2026-03-02 11:28:49,tahan õppida masinõpet,👎 Halb,Otsing leidis valed ained (RAG vektorotsing)
1,2026-03-02 11:30:28,otsin andmeteaduse kursust,👍 Hea,—
2,2026-03-02 11:33:54,tahan õppida veebiarendust,👍 Hea,—
3,2026-03-02 11:37:16,soovitan inglisekeelseid programmeerimiskursusui,👎 Halb,LLM hallutsineeris/vastas valesti (LLM generee...
4,2026-03-02 11:39:36,soovitan inglisekeelseid programmeerimiskursusui,👎 Halb,LLM hallutsineeris/vastas valesti (LLM generee...
5,2026-03-02 11:40:55,füüsika kursused,👎 Halb,Filtrid olid liiga karmid/valed (metaandmete f...
6,2026-03-02 11:42:32,kvantatarvutused ja kvantfüüsika,👍 Hea,—
7,2026-03-02 11:43:35,recommend me cooking classes,👎 Halb,LLM hallutsineeris/vastas valesti (LLM generee...


## 1. Üldstatistika

In [5]:
total = len(df)
good = (df['Hinnang'] == '👍 Hea').sum()
bad = (df['Hinnang'] == '👎 Halb').sum()

print(f'Kokku päringuid:   {total}')
print(f'Head vastused:     {good} ({good/total*100:.0f}%)')
print(f'Halvad vastused:   {bad} ({bad/total*100:.0f}%)')

Kokku päringuid:   8
Head vastused:     3 (38%)
Halvad vastused:   5 (62%)


## 2. Vigade analüüsi tabel

Vahesammud:
1. **Metaandmete filtreerimine** — filtrid on liiga karmid või valed
2. **RAG vektorotsing** — otsing leidis valed ained
3. **LLM genereerimine** — mudel hallutsineeris või ignoreeris konteksti

In [6]:
bad_df = df[df['Hinnang'] == '👎 Halb'].copy()

# Lühendame veatüübi nimetused
def lyhenda(x):
    if 'Filtrid' in str(x):
        return 'Metaandmete filtreerimine'
    elif 'RAG' in str(x) or 'Otsing' in str(x):
        return 'RAG vektorotsing'
    elif 'LLM' in str(x) or 'hallutsineeris' in str(x):
        return 'LLM genereerimine'
    return str(x)

bad_df['Vahesamm'] = bad_df['Veatüüp'].apply(lyhenda)

# Vigade tabel
error_table = bad_df[['Aeg', 'Kasutaja päring', 'Filtrid', 'Vahesamm']].copy()
error_table.columns = ['Aeg', 'Päring', 'Kasutatud filtrid', 'Süüdi olev vahesamm']
error_table = error_table.reset_index(drop=True)
error_table.index += 1
print('=== VIGADE ANALÜÜSI TABEL ===')
error_table

=== VIGADE ANALÜÜSI TABEL ===


,Aeg,Päring,Kasutatud filtrid,Süüdi olev vahesamm
1,2026-03-02 11:28:49,tahan õppida masinõpet,"Semester:sügis, EAP:6, Õppeaste:bakalaureuseõp...",RAG vektorotsing
2,2026-03-02 11:37:16,soovitan inglisekeelseid programmeerimiskursusui,"Semester:(kõik), EAP:(kõik), Õppeaste:bakalaur...",LLM genereerimine
3,2026-03-02 11:39:36,soovitan inglisekeelseid programmeerimiskursusui,"Semester:(kõik), EAP:(kõik), Õppeaste:bakalaur...",LLM genereerimine
4,2026-03-02 11:40:55,füüsika kursused,"Semester:kevad, EAP:1, Õppeaste:bakalaureuseõp...",Metaandmete filtreerimine
5,2026-03-02 11:43:35,recommend me cooking classes,"Semester:(kõik), EAP:(kõik), Õppeaste:(kõik), ...",LLM genereerimine


## 3. Vigade jaotus vahesammude kaupa

In [7]:
vahesammud = ['Metaandmete filtreerimine', 'RAG vektorotsing', 'LLM genereerimine']
counts = bad_df['Vahesamm'].value_counts().reindex(vahesammud, fill_value=0)

summary = pd.DataFrame({
    'Vahesamm': vahesammud,
    'Vigade arv': counts.values,
    'Protsent kõikidest vigastest päringutest': [f'{v/len(bad_df)*100:.0f}%' for v in counts.values]
})
summary = summary.set_index('Vahesamm')
print('=== VIGADE JAOTUS ===')
print(summary.to_string())

=== VIGADE JAOTUS ===
                           Vigade arv Protsent kõikidest vigastest päringutest
Vahesamm                                                                      
Metaandmete filtreerimine           1                                      20%
RAG vektorotsing                    1                                      20%
LLM genereerimine                   3                                      60%


In [10]:

summary['Vigade arv'] = counts.values
summary['Protsent'] = [f'{v/len(bad_df)*100:.0f}%' for v in counts.values]

print('=== VIGADE JAOTUS VAHESAMMUDE KAUPA ===\n')
print(summary.to_string())

print('\n=== VISUAALNE JAOTUS (tekstina) ===')
for samm, row in summary.iterrows():
    arv = row['Vigade arv']
    protsent = int(arv / len(bad_df) * 100)
    bar = '█' * protsent + '░' * (100 - protsent)
    print(f'{samm:<30} {bar[:30]} {arv} viga ({protsent}%)')

=== VIGADE JAOTUS VAHESAMMUDE KAUPA ===

                           Vigade arv Protsent kõikidest vigastest päringutest Protsent
Vahesamm                                                                               
Metaandmete filtreerimine           1                                      20%      20%
RAG vektorotsing                    1                                      20%      20%
LLM genereerimine                   3                                      60%      60%

=== VISUAALNE JAOTUS (tekstina) ===
Metaandmete filtreerimine      ████████████████████░░░░░░░░░░ 1 viga (20%)
RAG vektorotsing               ████████████████████░░░░░░░░░░ 1 viga (20%)
LLM genereerimine              ██████████████████████████████ 3 viga (60%)


## 4. Järeldused

| Vahesamm | Vigade arv | % vigastest |
|---|---|---|
| Metaandmete filtreerimine | 1 | 20% |
| RAG vektorotsing | 1 | 20% |
| LLM genereerimine | 3 | 60% |

**Peamised tähelepanekud:**

1. **LLM genereerimine on kõige suurem probleemide allikas (60% vigadest).** Mudel kaldus ignoreerima RAG konteksti ja hallutsineerima väliseid ressursse (Coursera, freeCodeCamp, kokakolid jne), eriti siis, kui päring ei sobinud hästi saadaolevate kursustega.

2. **Metaandmete filtreerimine põhjustas 1 vea (20%).** Liiga range filter (EAP=1) jättis andmestiku peaaegu tühjaks, kuid LLM vastas ikkagi väljamõeldud kursustega.

3. **RAG vektorotsing põhjustas 1 vea (20%).** Masinõppe päringu puhul leiti RAGiga ebasobivad kursused (arvutiarhitektuur, biomeditsiinitehnika), mis ei vastanud kasutaja tegelikule vajadusele.

**Soovitused parandamiseks:**
- Lisada süsteemiviipa karm piirang: "Ära soovita ühtegi kursust, mida pole antud nimekirjas"
- Kui filtreerimise järel jääb alla 3 kursuse, hoiatada kasutajat ja laiendada filtreid automaatselt
- Kaaluda reranking-meetodeid RAG otsingu täpsuse parandamiseks

In [9]:
testjuhtumid = [
    {
        'nr': 1,
        'päring': 'tahan õppida andmeteadust',
        'filtrid': 'Semester: sügis, EAP: 6, Õppeaste: bakalaureuseõpe',
        'oodatavad_ID': ['SVUH.00.059', 'SVNC.00.307'],
        'põhjendus': 'Andmepädevus ja Andmebaasisüsteemid on otseselt andmeteadusega seotud'
    },
    {
        'nr': 2,
        'päring': 'tahan õppida veebiarendust',
        'filtrid': 'Filtrid puuduvad',
        'oodatavad_ID': ['LTAT.03.015', 'MTAT.03.297'],
        'põhjendus': 'Veebilehtede loomine (edasijõudnutele) on otseselt veebiarenduse kursused'
    },
    {
        'nr': 3,
        'päring': 'kvantatarvutused ja kvantfüüsika',
        'filtrid': 'Filtrid puuduvad',
        'oodatavad_ID': ['MTAT.07.024', 'LOFY.04.073', 'LTFY.04.012'],
        'põhjendus': 'Kvantkrüptograafia, Kvantmehaanika ja Kvantarvutuse alused on otseselt seotud'
    },
    {
        'nr': 4,
        'päring': 'tahan õppida programmeerimist inglise keeles',
        'filtrid': 'Keel: inglise keel, Õppeaste: bakalaureuseõpe',
        'oodatavad_ID': ['LTAT.03.001_1', 'LOTI.05.100'],
        'põhjendus': 'Programmeerimise kursused inglise keeles bakalaureuseõppes'
    },
    {
        'nr': 5,
        'päring': 'tahan õppida masinõpet',
        'filtrid': 'Semester: sügis, EAP: 6, Õppeaste: bakalaureuseõpe',
        'oodatavad_ID': ['LTAT.03.001_1', 'LTMS.00.081', 'LOTI.05.100'],
        'põhjendus': 'Programmeerimine ja matemaatika on masinõppe eeldusained'
    }
]

test_df = pd.DataFrame(testjuhtumid)
test_df.columns = ['Nr', 'Päring', 'Filtrid', 'Oodatavad unique_ID-d', 'Põhjendus']
test_df = test_df.set_index('Nr')
print('=== TESTJUHTUMID ===')
test_df

=== TESTJUHTUMID ===


,Päring,Filtrid,Oodatavad unique_ID-d,Põhjendus
Nr,,,,
1,tahan õppida andmeteadust,"Semester: sügis, EAP: 6, Õppeaste: bakalaureus...","[SVUH.00.059, SVNC.00.307]",Andmepädevus ja Andmebaasisüsteemid on otsesel...
2,tahan õppida veebiarendust,Filtrid puuduvad,"[LTAT.03.015, MTAT.03.297]",Veebilehtede loomine (edasijõudnutele) on otse...
3,kvantatarvutused ja kvantfüüsika,Filtrid puuduvad,"[MTAT.07.024, LOFY.04.073, LTFY.04.012]","Kvantkrüptograafia, Kvantmehaanika ja Kvantarv..."
4,tahan õppida programmeerimist inglise keeles,"Keel: inglise keel, Õppeaste: bakalaureuseõpe","[LTAT.03.001_1, LOTI.05.100]",Programmeerimise kursused inglise keeles bakal...
5,tahan õppida masinõpet,"Semester: sügis, EAP: 6, Õppeaste: bakalaureus...","[LTAT.03.001_1, LTMS.00.081, LOTI.05.100]",Programmeerimine ja matemaatika on masinõppe e...
